# 01 – Download NSINA slice (STGWKS-13)

**One-time setup before running:**
1. Log in at huggingface.co, open `sinhala-nlp/NSINA` and accept the access conditions.
2. Create a **read** token: huggingface.co → Settings → Access Tokens.
3. In Colab, open the 🔑 Secrets panel (left sidebar), add a secret named `HF_TOKEN` with that token, and allow notebook access.

A GPU is not needed. The file (~1.9 GB) is read **one article at a time**, so memory stays low.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys, json
DATA_DIR = '/content/drive/MyDrive/sinhala-tiny-gpt'
REPO = '/content/sinhala-tiny-gpt'
if not os.path.exists(REPO):
    !git clone https://github.com/Amitha4469/sinhala-tiny-gpt.git {REPO}
else:
    !git -C {REPO} pull
sys.path.insert(0, REPO)

!pip install -q ijson
import ijson
print('ijson backend:', ijson.backend)  # yajl2_c is the fast one

In [ ]:
from sgpt.data.download import download_nsina, iter_records, make_slice, summarize, save_jsonl

# Downloads ~1.9 GB the first time; reuses the cached file afterwards
paths = download_nsina(token=userdata.get('HF_TOKEN'))
for p in paths:
    print(p, round(os.path.getsize(p) / 1e9, 2), 'GB')

In [ ]:
# Stream all articles and keep a seeded random 40 MB slice (a few minutes)
rows = (r for p in paths for r in iter_records(p))
records, seen = make_slice(rows, target_mb=40, seed=1234, progress_every=50_000)

summary = summarize(records)
summary['rows_in_nsina'] = seen
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
out_path = f'{DATA_DIR}/raw/nsina_slice_40mb_seed1234.jsonl'
save_jsonl(records, out_path)
with open(f'{DATA_DIR}/raw/nsina_slice_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('Saved', len(records), 'articles to', out_path)